# Preparando dataframe, as atividades estarão mais abaixo.

In [1]:
import pandas as pd

url = "https://servicodados.ibge.gov.br/api/v3/agregados/4093/periodos/201201-202602/variaveis/4099?localidades=N3[26]&classificacao=2[all]"

data = pd.read_json(url)

data

,id,variavel,unidade,resultados
0,4099,"Taxa de desocupação, na semana de referência, ...",%,"[{'classificacoes': [{'id': '2', 'nome': 'Sexo..."


In [3]:
for i, resultado in enumerate(data["resultados"][0]):
    print(i, resultado["classificacoes"][0]["categoria"])

0 {'6794': 'Total'}
1 {'4': 'Homens'}
2 {'5': 'Mulheres'}


In [5]:
serie_total = data["resultados"][0][0]["series"][0]["serie"]
serie_homens = data["resultados"][0][1]["series"][0]["serie"]
serie_mulheres = data["resultados"][0][2]["series"][0]["serie"]

In [7]:
df = pd.DataFrame({
    "total": pd.Series(serie_total),
    "homens": pd.Series(serie_homens),
    "mulheres": pd.Series(serie_mulheres)
})

df.index.name = "periodo"
df = df.reset_index()

df.head()

,periodo,total,homens,mulheres
0,201201,9.6,7.1,13.1
1,201202,8.3,6.4,11.0
2,201203,9.4,7.6,11.8
3,201204,9.2,7.9,11.1
4,201301,10.7,8.2,14.2


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   periodo   58 non-null     object
 1   total     58 non-null     object
 2   homens    58 non-null     object
 3   mulheres  58 non-null     object
dtypes: object(4)
memory usage: 1.9+ KB


In [11]:
df["total"] = df["total"].replace("...", "0")
df["homens"] = df["homens"].replace("...", "0")
df["mulheres"] = df["mulheres"].replace("...", "0")

In [13]:
df["total"] = df["total"].astype(float)
df["homens"] = df["homens"].astype(float)
df["mulheres"] = df["mulheres"].astype(float)

In [15]:
df["ano"] = df["periodo"].str[:4]
df["tri"] = df["periodo"].str[-2:].astype(int)

df["periodo"] = pd.PeriodIndex(
    df["ano"] + "Q" + df["tri"].astype(str),
    freq="Q"
)

df["periodo"] = df["periodo"].dt.to_timestamp()

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58 entries, 0 to 57
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   periodo   58 non-null     datetime64[ns]
 1   total     58 non-null     float64       
 2   homens    58 non-null     float64       
 3   mulheres  58 non-null     float64       
 4   ano       58 non-null     object        
 5   tri       58 non-null     int64         
dtypes: datetime64[ns](1), float64(3), int64(1), object(1)
memory usage: 2.8+ KB


In [20]:
df.head()

,periodo,total,homens,mulheres,ano,tri
0,2012-01-01,9.6,7.1,13.1,2012,1
1,2012-04-01,8.3,6.4,11.0,2012,2
2,2012-07-01,9.4,7.6,11.8,2012,3
3,2012-10-01,9.2,7.9,11.1,2012,4
4,2013-01-01,10.7,8.2,14.2,2013,1


# Média

In [22]:
print("Média - Total:", df["total"].mean())
print("Média - Mulheres:", df["mulheres"].mean())
print("Média - Homens:", df["homens"].mean())

Média - Total: 10.65
Média - Mulheres: 12.505172413793103
Média - Homens: 9.28448275862069


# Média retirando períodos sem dados

In [28]:
df_filtrado = df.query(
    "total != 0 or mulheres != 0 or homens != 0"
)
df_filtrado

,periodo,total,homens,mulheres,ano,tri
0,2012-01-01,9.6,7.1,13.1,2012,1
1,2012-04-01,8.3,6.4,11.0,2012,2
2,2012-07-01,9.4,7.6,11.8,2012,3
3,2012-10-01,9.2,7.9,11.1,2012,4
4,2013-01-01,10.7,8.2,14.2,2013,1
5,2013-04-01,9.7,8.3,11.7,2013,2
6,2013-07-01,8.5,6.6,11.2,2013,3
7,2013-10-01,7.4,6.1,9.3,2013,4
8,2014-01-01,8.8,7.4,10.9,2014,1
9,2014-04-01,8.0,7.2,9.2,2014,2


# Mediana

In [30]:
print("Média - Total:", df_filtrado["total"].mean())
print("Média - Mulheres:", df_filtrado["mulheres"].mean())
print("Média - Homens:", df_filtrado["homens"].mean())

Média - Total: 12.354000000000001
Média - Mulheres: 14.505999999999998
Média - Homens: 10.77


In [32]:
print("Mediana - Total:", df["total"].median())
print("Mediana - Mulheres:", df["mulheres"].median())
print("Mediana - Homens:", df["homens"].median())

Mediana - Total: 10.899999999999999
Mediana - Mulheres: 13.05
Mediana - Homens: 9.5


In [34]:
print("Mediana - Total:", df_filtrado["total"].median())
print("Mediana - Mulheres:", df_filtrado["mulheres"].median())
print("Mediana - Homens:", df_filtrado["homens"].median())

Mediana - Total: 11.8
Mediana - Mulheres: 14.3
Mediana - Homens: 10.149999999999999


# Comparação da média e mediana

In [36]:
print("TOTAL")
print("Média:", df_filtrado["total"].mean())
print("Mediana:", df_filtrado["total"].median())

print()

print("HOMENS")
print("Média:", df_filtrado["homens"].mean())
print("Mediana:", df_filtrado["homens"].median())

print()

print("MULHERES")
print("Média:", df_filtrado["mulheres"].mean())
print("Mediana:", df_filtrado["mulheres"].median())

TOTAL
Média: 12.354000000000001
Mediana: 11.8

HOMENS
Média: 10.77
Mediana: 10.149999999999999

MULHERES
Média: 14.505999999999998
Mediana: 14.3


# Agregação temporal por ano

In [42]:
media_anual = (
    df_filtrado
    .set_index("periodo")
    .resample("YE")
    .mean(numeric_only=True)
)

media_anual

,total,homens,mulheres,tri
periodo,,,,
2012-12-31,9.125,7.250000,11.750000,2.5
2013-12-31,9.075,7.300000,11.600000,2.5
2014-12-31,8.250,7.175000,9.775000,2.5
2015-12-31,9.950,8.875000,11.450000,2.5
2016-12-31,14.750,13.450000,16.600000,2.5
2017-12-31,17.850,16.450000,19.750000,2.5
2018-12-31,16.900,15.950000,18.200000,2.5
2019-12-31,15.650,13.575000,18.400000,2.5
2020-12-31,14.800,13.400000,16.700000,1.0


# Moda

In [44]:
print("Moda - Total:")
print(df["total"].mode())

print()

print("Moda - Mulheres:")
print(df["mulheres"].mode())

print()

print("Moda - Homens:")
print(df["homens"].mode())

Moda - Total:
0    0.0
Name: total, dtype: float64

Moda - Mulheres:
0    0.0
Name: mulheres, dtype: float64

Moda - Homens:
0    0.0
Name: homens, dtype: float64


In [46]:
print("Moda - Total:")
print(df_filtrado["total"].mode())

print()

print("Moda - Mulheres:")
print(df_filtrado["mulheres"].mode())

print()

print("Moda - Homens:")
print(df_filtrado["homens"].mode())

Moda - Total:
0     9.2
1    14.2
Name: total, dtype: float64

Moda - Mulheres:
0    10.4
1    11.1
2    11.8
3    12.9
4    18.6
Name: mulheres, dtype: float64

Moda - Homens:
0     6.8
1     7.4
2     8.2
3     8.3
4    10.0
5    10.3
6    11.0
7    12.5
8    17.4
Name: homens, dtype: float64


# Resumo final das medidas

In [48]:
print("===== TOTAL =====")
print("Média:", df_filtrado["total"].mean())
print("Mediana:", df_filtrado["total"].median())
print("Moda:")
print(df_filtrado["total"].mode())

print()

print("===== HOMENS =====")
print("Média:", df_filtrado["homens"].mean())
print("Mediana:", df_filtrado["homens"].median())
print("Moda:")
print(df_filtrado["homens"].mode())

print()

print("===== MULHERES =====")
print("Média:", df_filtrado["mulheres"].mean())
print("Mediana:", df_filtrado["mulheres"].median())
print("Moda:")
print(df_filtrado["mulheres"].mode())

===== TOTAL =====
Média: 12.354000000000001
Mediana: 11.8
Moda:
0     9.2
1    14.2
Name: total, dtype: float64

===== HOMENS =====
Média: 10.77
Mediana: 10.149999999999999
Moda:
0     6.8
1     7.4
2     8.2
3     8.3
4    10.0
5    10.3
6    11.0
7    12.5
8    17.4
Name: homens, dtype: float64

===== MULHERES =====
Média: 14.505999999999998
Mediana: 14.3
Moda:
0    10.4
1    11.1
2    11.8
3    12.9
4    18.6
Name: mulheres, dtype: float64
